# 8.16 範圍、格式與結束，如何各自教與檢查？


書店的告示仍是四行：第1列「今日休館」、第2列「Closed today」、第3列「明日開放」、第4列「Open tomorrow」。現在收到的完整要求是：「只抄兩行中文，依原順序，回一個只有text欄位的JSON物件，欄位值是文字清單；JSON外不要加字。」

一份合格的可見回答如下，之後還要正常結束：

```json
{"text":["今日休館","明日開放"]}
```

先看文字內容：兩句應抄的中文有沒有錯字、漏字或交換順序？再看範圍：兩句是不是都交了，有沒有多抄英文，清單裡有沒有混進其他說明？格式另問三件事：整份文字是否就是**裸JSON**，也就是JSON物件前後沒有其他文字；是否只有text欄位；值是否為字串清單。這裡的格式規則不靠「清單一定兩項」代替範圍檢查；多抄四行仍可能是格式正確的清單。

多出來的字要記在哪一欄，看它是什麼、放在哪裡。要求寫了「JSON外不要加字」，這是格式要求：JSON前面多一句「以下是結果：」，或物件裡多一個欄位，都記在格式；把說明當成一項放進清單，是多交了要求以外的項目，記在範圍。[8.14](https://birdhackor.github.io/tiny-perceptron-vlm/8.14.html)沒有要求JSON，多出的說明只能算範圍；這裡要求了裸JSON，JSON外的說明就由格式負責。多抄的那兩句英文則不論放在清單裡、JSON後面或另一個欄位，都記在範圍；不在清單裡時，格式也同時未通過。

最後核對正常結束。它不是檢查最後一個可見字是否為右大括號。需要查看模型實際生成的結束token與停止原因，確認回答之後由模型選出EOS，而非上限用完或程式強制停下。若EOS提早出現，內容可能尚未完成；若文字都已寫完但沒有EOS，這次也不能記成主動結束。

圖中的四張檢查卡都指向同一題。每張卡只回答自己的問題，最下面要求四項同時通過；它畫的是驗收關係，不是在宣稱模型內有四個獨立的部件。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-08-new-8.16-joint-check.svg")))

下面是一份正確範例與五份反例，每份都附上停止紀錄。「內容」這欄只核對兩句需要抄的中文，額外英文另記在範圍；「正常EOS」是這個紙上紀錄給定的條件，不是模型實測成績。

| 手寫回答／停止條件 | 內容 | 範圍 | 格式 | 正常EOS |
| --- | --- | --- | --- | --- |
| 正確JSON，隨後選EOS | 通過 | 通過 | 通過 | 通過 |
| 把第一句寫成「今日開館」，其他與正確JSON相同，隨後選EOS | 未通過 | 通過 | 通過 | 通過 |
| JSON的清單照抄全部四行，隨後選EOS | 通過 | 未通過 | 通過 | 通過 |
| 只給兩行正確中文普通文字，隨後選EOS | 通過 | 通過 | 未通過 | 通過 |
| 前面多一句「以下是結果：」，再接正確JSON，隨後選EOS | 通過 | 通過 | 未通過 | 通過 |
| 可見文字是正確JSON，因生成上限而停，沒有EOS | 通過 | 通過 | 通過 | 未通過 |

第二份的抄寫範圍仍是兩句中文，但第一句有錯字；第三份沒有漏掉應抄的兩句，卻多抄英文；第五份的兩句與清單都對，只是JSON外多了一句，錯在格式。把它們分項記錄，就能知道是內容錯了，還是選取範圍不符，而不必把兩者都叫作「格式錯」。真實回答也可能一次違反兩項以上；本表故意一次改一件事，讓差別容易看見。

這張表也能交給程式核對。下面的`check`收下一份回答與它的停止原因，分別回答四個問題。


In [ ]:
import json

target = ["今日休館", "明日開放"]
excluded = ["Closed today", "Open tomorrow"]


def check(reply, stop):
    try:
        data = json.loads(reply)
    except json.JSONDecodeError:
        data = None
    form = (
        isinstance(data, dict)
        and set(data) == {"text"}
        and isinstance(data["text"], list)
        and all(isinstance(item, str) for item in data["text"])
    )
    found = [reply.find(line) for line in target]
    content = -1 not in found and found == sorted(found)
    scope = not any(line in reply for line in excluded)
    if form:
        scope = scope and len(data["text"]) == len(target)
    return {"內容": content, "範圍": scope, "格式": form, "正常EOS": stop == "EOS"}

`target`是兩句應抄的中文，`excluded`是這次不該抄的兩句英文。`try`／`except`、`isinstance`與`set(data) == {"text"}`的用法同[8.5](https://birdhackor.github.io/tiny-perceptron-vlm/8.5.html)：解析失敗時`data`設成`None`，格式就不會通過。`form`依序要求讀到字典、只有text欄位、值是清單，而且清單裡每一項都是字串；`all(...)`只在每一項都成立時才是True。

內容與範圍先不管格式，直接在整份回答裡找字。`reply.find(line)`回傳這一句在回答裡第一次出現的位置，找不到時回傳`-1`。`content`要求兩句都找得到，而且位置由前到後：`sorted(found)`把位置由小排到大，排過以後和原來一樣，就表示順序沒有顛倒。有錯字的那一句找不到，內容就不通過；多抄的英文不影響這一欄。`scope`先用`any(...)`查有沒有任何一句英文出現在回答裡：`any`只要有一項成立就是True，前面加`not`就是「一句都沒有」。格式通過、讀得到清單時，`scope`再要求清單恰好兩項，只漏交一句或只多放一項都會被抓到。`stop`是表中給定的停止原因；真正評估時，它要從生成紀錄讀出模型是選了EOS，還是被上限截停（[7.9](https://birdhackor.github.io/tiny-perceptron-vlm/7.9.html)）。


In [ ]:
cases = [
    ('{"text":["今日休館","明日開放"]}', "EOS"),
    ('{"text":["今日開館","明日開放"]}', "EOS"),
    ('{"text":["今日休館","Closed today","明日開放","Open tomorrow"]}', "EOS"),
    ("今日休館\n明日開放", "EOS"),
    ('以下是結果：{"text":["今日休館","明日開放"]}', "EOS"),
    ('{"text":["今日休館","明日開放"]}', "上限"),
]
for reply, stop in cases:
    result = check(reply, stop)
    print(result, "全部通過", all(result.values()))

六行輸出依序對應表格的六列，四項的True與False應與表格逐格相同。`all(result.values())`把整題成功寫成「四項都通過」：只有第一份是True，其餘五份各有一項未通過，整題就不算完成。

本節正文還提到幾種回答，下面把它們交給同一個`check`，停止原因都給定為EOS。其中一份由`json.dumps`產生：它是`json.loads`的反方向，把Python資料寫成JSON字串；程式先印出它用預設設定寫出的正確答案。


In [ ]:
encoded = json.dumps({"text": ["今日休館", "明日開放"]})
print(encoded)
more = [
    ("JSON後接英文", '{"text":["今日休館","明日開放"]} Closed today'),
    ("en欄位放英文", '{"text":["今日休館","明日開放"],"en":["Closed today","Open tomorrow"]}'),
    ("說明放進清單", '{"text":["今日休館","明日開放","以下是結果"]}'),
    ("清單只交一句", '{"text":["今日休館"]}'),
    ("目標句在說明裡", '以下不是今日休館：{"text":["今日開館","明日開放"]}'),
    ("json.dumps預設", encoded),
    ("普通文字漏一句加說明", "今日休館\n以下是結果"),
    ("清單漏一句加說明", '{"text":["今日休館","以下是結果"]}'),
]
for name, reply in more:
    print(name, check(reply, "EOS"))

第一行的中文都寫成了`\u4eca`這類代號。接下來兩行對應「多出來的字要記在哪一欄」那段：英文接在JSON後面或放進另一個欄位，範圍與格式都未通過。再兩行是`scope`數項數的例子：說明放進清單、清單只交一句，格式都通過，範圍都未通過。最後四行是下面「規則也有它的邊界」那段要談的盲點。

替這份任務準備示範時，同樣要把四項都寫進完整目標：兩句原文、限定範圍、指定JSON，以及答完之後的EOS。修正錯字不能替多抄英文解圍，補上JSON也不能替被截掉的EOS補回監督。先找出哪一項不符，再回到上一節的請求與目標配對逐筆修正。

整題成功不是四欄取平均。「三項通過」仍有一項必要要求沒有完成。若一次檢查多道題，應把每項通過題數和全部通過題數各自記錄，分母都包含那一批原定題目；不能先丟掉被上限截停的回答，再替剩下的答案算成功率。

規則也有它的邊界。能被JSON讀取，不等於只有指定欄位；搜尋到「今日休館」，不等於沒多抄英文；結尾出現某句文字，也不能證明模型正常選EOS。`check`的內容與範圍靠找字，也有自己的盲點：目標句若剛好出現在一句說明裡，例如前面寫「以下不是今日休館：」、清單裡卻寫成「今日開館」，內容會被誤判為通過；反過來，正確答案若由`json.dumps`用預設設定寫出，中文會變成`\u4eca`（代表「今」）這類代號，回答裡找不到「今日休館」，內容會被誤判為未通過，格式卻通過。`excluded`只認得這兩句英文。格式不通過、讀不到清單時，範圍只能查有沒有這兩句英文，數不出漏交或多交的項目，這時漏抄的一句要靠內容欄發現；讀得到清單時，範圍對清單也只數項數，漏掉「明日開放」又多放一句說明，清單仍是兩項，範圍照樣通過，只有內容未通過。換一份告示就得重寫這些規則。因此要先寫清楚接收端需要什麼，再選相應檢查。這個抄寫任務可以精確比對；換成開放式建議，多種說法都可能合理，不能把本節的字串規則直接當成通用內容裁判。

練習把要求改成「只抄英文，其餘格式相同」：把第一段程式的`target`改成`["Closed today", "Open tomorrow"]`，`excluded`改成`["今日休館", "明日開放"]`。看第二段程式的第一份回答，也就是原先那份中文JSON，先預測哪些項目仍通過：格式和正常EOS仍是True，內容與範圍都變成False。最後把這份回答換成英文目標`{"text":["Closed today","Open tomorrow"]}`，四項應全部通過，確認不是只改題目、卻拿舊答案當新題的成功示範。

<details>
<summary>補充：聯合驗收與IFEval的檢查器界線</summary>

[IFEval原論文](https://arxiv.org/pdf/2311.07911v1)§3區分逐項通過與同題全部要求通過；Table 1、§2.2說明可檢查要求與strict／loose評分。其[官方實作](https://github.com/google-research/google-research/blob/e49bbfe381c9c0e564b937f1c4e163a2273c65cc/instruction_following_eval/instructions.py)的JSON規則允許移除指定的Markdown外框，其他規則也有各自的判準。它們不等於本節「裸JSON、只有text欄位、值為字串清單」的全部要求，也不檢查模型是否自然選出EOS。

正常結束另依[Transformers v4.57.1的生成設定](https://github.com/huggingface/transformers/blob/v4.57.1/src/transformers/generation/configuration_utils.py)區分eos_token_id、forced_eos_token_id與長度上限。相關段落與適用界線見[指令遵循來源筆記](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-revision-20261005/sources/instruction-following.md)。

</details>
